# Exploration des données Orsay (arrêté du 25/09/2026)
Prérequis : `uv run python exploration/prepare_data.py` génère `clean/`. Quantités relues en `Decimal`.

In [ ]:
import subprocess
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd

ICI = Path.cwd() if Path.cwd().name == "exploration" else Path.cwd() / "exploration"
subprocess.run([sys.executable, str(ICI / "prepare_data.py")], check=True)
CLEAN = ICI / "clean"
QTE = ["quantite", "PMS_A", "PMS_B", "PMS_C", "A", "B", "C", "ecart_A", "ecart_B", "ecart_C"]


def lire(nom):
    df = pd.read_csv(CLEAN / nom, sep=";", dtype=str, keep_default_na=False)
    for c in df.columns.intersection(QTE):
        df[c] = df[c].map(Decimal)
    return df


pos, journal, comp = (
    lire("positions_unifiees.csv"),
    lire("journal_nettoyage.csv"),
    lire("comparatif.csv"),
)

## 1. Volumétrie : lignes par source et statut

In [ ]:
pd.crosstab(pos.source, pos.statut, margins=True)

## 2. Journal de nettoyage : quelles transformations, combien

In [ ]:
journal.groupby(["niveau", "fichier", "regle"]).size().rename("n").reset_index()

In [ ]:
journal[journal.niveau != "NORMALISE"]  # exclusions, signalements, bloquants

In [ ]:
journal[journal.niveau == "NORMALISE"].groupby("regle").head(2)  # exemples brut → retenu

## 3. Positions unifiées (format commun)

In [ ]:
pos

## 4. Comparatif PMS ↔ conservateurs par (fonds, ISIN)

In [ ]:
comp  # trié : lignes non rapprochées en tête

In [ ]:
comp[comp.rapproche == "False"]

## 5. Base orsay.db (information, lecture seule)

In [ ]:
db = {
    n: pd.read_csv(CLEAN / f"db_{n}.csv", sep=";", dtype=str, keep_default_na=False)
    for n in ["portefeuilles", "titres", "cours", "positions", "mouvements"]
}
{n: len(d) for n, d in db.items()}

In [ ]:
db["titres"].groupby(["type_titre", "mode_cotation"]).size()

In [ ]:
db["mouvements"][
    db["mouvements"].statut != "DENOUE"
]  # opérations en attente → explication des écarts QTE

In [ ]:
db["positions"]